# Ollama Raw LLM Completion


## Setup

Run Ollama before executing cells:
- `ollama serve`
- `ollama pull gemma3:4b` (or another <=12B local model)

## Ollama API

In [ ]:
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import StreamingResponse
import json
import uvicorn
import threading
from Modules.conversion_1.conversion import messages_to_prompt
from Modules.conversion_1.prompt_factor import PromptBuilder

# API Setting
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:4b"

app = FastAPI(title="HKBU Study Companion")
builder = PromptBuilder()

class Query(BaseModel):
    """HKBU Study Assistant providing three mutually exclusive working modes.
    
    ====================   ===========   ===============   =====================
    Mode                    is_search     is_study_plan     Optional Features
    ====================   ===========   ===============   =====================
    Normal Mode             False         False             think_mode
    Search Mode (RAG)       True          False             use_embedding_retrieval
    Study Plan Mode         False         True              -
    ====================   ===========   ===============   =====================
    
    Constraints:
        - is_search and is_study_plan cannot both be True
        - use_embedding_retrieval only takes effect when is_search=True
        - think_mode only takes effect when is_search=False and is_study_plan=False
    """

    question: str                           # user query
    context: list = []                      # history record
    system_prompt: str = ""                 # system role setting
    is_search: bool = False                 # enable search mode?
    use_embedding_retrieval: bool = False   # using embedding retrieval?(if not, use lexical retrieval)
    think_mode: bool = False                # enable think mode?
    is_study_plan: bool = False             # ebable study plan mode?

def complete_document(prompt: str):
    """Generate streaming response from Ollama"""
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": True,
        "options": {
            "num_predict": 2048,
            "temperature": 0.3
        }
    }
    
    try:
        with requests.post(OLLAMA_URL, json=payload, stream=True, timeout=120) as r:
            r.raise_for_status()
            for line in r.iter_lines(decode_unicode=True):
                if not line:
                    continue
                    
                try:
                    data = json.loads(line)
                    token = data.get("response", "")
                    if token:
                        yield token
                    
                    if data.get("done", False):
                        break
                        
                except json.JSONDecodeError:
                    print(f"Failed to parse JSON: {line}")
                    continue
                    
    except requests.exceptions.ConnectionError:
        yield "\nError: Cannot connect to Ollama server. Please ensure Ollama is running."
    except requests.exceptions.Timeout:
        yield "\nError: Request timeout after 120 seconds."
    except Exception as e:
        yield f"\nError: {str(e)}"

# API
@app.post("/ask")
def ask(q: Query):

    full_prompt_list = builder.get_full_prompt_list(
        question=q.question,
        context = q.context,
        system_prompt = q.system_prompt,
        is_search = q.is_search,
        use_embedding_retrieval= q.use_embedding_retrieval,
        think_mode = q.think_mode,
        is_study_plan = q.is_study_plan
    )

    full_prompt = messages_to_prompt(full_prompt_list)
    return StreamingResponse(
        complete_document(full_prompt),
        media_type="text/plain; charset=utf-8"
    )

✅ Backend Address: http://localhost:8000
✅ Backend Document: http://localhost:8000/docs


INFO:     Started server process [27448]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): 通常每个套接字地址(协议/网络地址/端口)只允许使用一次。
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:63335 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:63335 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:50974 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:50974 - "GET /openapi.json HTTP/1.1" 200 OK


In [ ]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8326, log_level="info")
    print("✅ Backend Address: http://localhost:8326")
    print("✅ Backend Document: http://localhost:8326/docs")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

## Test

Here are the testing functions for each function of the model

In [ ]:
from textwrap import dedent

def test_ask_directly(test_cases):

    for i, test_case in enumerate(test_cases):
        print(f"\n{'='*60}")
        print(f"test {i+1}: {test_case.get('description', '')}")
        print(f"question: {test_case['question']}")
        print(f"{'='*60}\n")
        
        query = Query(**test_case)
        
        response = ask(query)
        
        print("answer: ", end="")
        for chunk in response.body_iterator:
            print(chunk, end="", flush=True)
        print("\n")

# test cases
TEST_CASES = [
    {
        "description": "Normal Mode(no thinking)",
        "question": "What's mechine learning?",
    },
    {
        "description": "Normal Mode(enable thinking)",
        "question": dedent("""
            A train travels from City A to City B at 60 km/h. 
            On the return trip from City B to City A, it travels at 40 km/h. 
            What is the average speed for the entire round trip? 
            Please solve step by step.
        """).strip(),
        "think_mode": True,
    },
    {
        "description": "Search Mode(lexical retrieval)",
        "question": "What are the assessment methods and weightings for the course COMP7025?",
        "think_mode": True,
    },
    {
        "description": "Search Mode(embedding retrieval)",
        "question": dedent("""
            How do students get evaluated in the AI for Digital Transformation class? 
            What portion is coursework versus the final exam?
        """).strip(),
        "think_mode": True,
        "use_embedding_retrieval": True,
    },
    {
        "description": "Study plan Mode",
        "question": dedent("""
        Please help me create a 14 day review plan. 
        I am only available in the evenings on weekdays, 
        but on weekends, I am available in the afternoons and evenings. 
        This plan starts from Monday
    """).strip(),
        "is_study_plan": True
    }
]

# test
test_ask_directly(TEST_CASES)

If you get `ConnectionError`, start Ollama first (`ollama serve`) and ensure your model is available (`ollama pull llama3.1:8b`).
If `requests` is missing, run `%pip install requests` in a notebook cell.
